In [ ]:
from pathlib import Path

import networkx as nx
import iplotx as ipx

import numpy as np
import polars as pl
import polars.selectors as cs

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

from climate_attitudes.settings import Config
from climate_attitudes.dataset import Dataset
from climate_attitudes.utils import (
    calculate_stationary_distribution,
    calculate_transition_probabilities,
    calculate_proportion_response,
    relative_entropy,
)
from climate_attitudes.correlation import pcorr
from climate_attitudes.visualisation import configure_mpl

configure_mpl(Path("../fonts"))

config = Config(_env_file="../.env")

data = Dataset.load(config)

In [ ]:
columns = {
    "Climate change happening": {
        "colname": "cc1",
        "responses": ["No", "Maybe", "Yes"],
    },
    "Climate change anthropogenic ('people should act')": {
        "colname": "cvcc4_should",
        "responses": [
            "Strongly disagree",
            "Disagree",
            "Neither agree nor disagree",
            "Agree",
            "Strongly agree",
        ],
    },
    "Climate change worry": {
        "colname": "cc6",
        "responses": [
            "Not at all worried",
            "Not very worried",
            "Somewhat worried",
            "Very worried",
        ],
    },
    "Future generation harm": {
        "colname": "cc5_world",
        "responses": [
            "Don't know",
            "Not at all",
            "Only a little",
            "A moderate amount",
            "A great deal",
        ],
    },
}

In [ ]:
def standardise(expr: pl.Expr | cs.Selector) -> pl.Expr:
    return (expr - expr.mean()) / expr.std()


pids = (
    data.participant.filter(wave_1=True, wave_2=True)
    .select("participant_id")
    .collect()
    .to_series()
    .implode()
)

resp = data.response.filter(
    pl.col("wave") <= 2, pl.col("participant_id").is_in(pids)
).select(
    "participant_id",
    pl.col("wave"),  # .replace_strict({1: "wave_1", 2: "wave_2"}),
    pl.col("cc1").replace({1: 2, 99: 1}),
    "cvcc4_should",
    "cc5_world",
    "cc6",
)


resp_with_dk = (
    resp.with_columns(pl.col("cc5_world").replace(99, 0))
    .with_columns(pl.len().over("participant_id").alias("n_waves"))
    .filter(n_waves=2)
    .drop("n_waves")
    .with_columns(standardise(cs.exclude("participant_id", "wave")))
    .collect()
)

resp_no_dk = (
    resp.with_columns(pl.col("cc5_world").replace(99, None))
    .filter(pl.all_horizontal(pl.all().is_not_null()))
    .with_columns(pl.len().over("participant_id").alias("n_waves"))
    .filter(n_waves=2)
    .drop("n_waves")
    .with_columns(standardise(cs.exclude("participant_id", "wave")))
    .collect()
)

## Pairwise correlation + partial correlation

In [ ]:
corr = resp_no_dk.filter(wave=2).drop("participant_id", "wave").to_pandas().corr()

partial_corr = pcorr(resp_no_dk.filter(wave=2).drop("participant_id", "wave"))

# Generate a mask for the upper triangle
mask = np.triu(np.ones_like(partial_corr, dtype=bool), k=1)

# Set up the matplotlib figure
fig, axes = plt.subplots(ncols=2, figsize=(6, 3), constrained_layout=True)

# Generate a custom diverging colormap
cmap = sns.diverging_palette(20, 230, as_cmap=True)
# cmap = "coolwarm"

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(
    corr,
    mask=mask,
    cmap=cmap,
    annot=True,
    fmt=".1f",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar=False,
    ax=axes[0],
)

# Draw the heatmap with the mask and correct aspect ratio
g = sns.heatmap(
    partial_corr,
    mask=mask,
    cmap=cmap,
    annot=True,
    fmt=".1f",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.675, "pad": 0.25},
    ax=axes[1],
)

labels = ["CC Happening", "CC Anthropogenic (proxy)", "CC Worry", "Future gen harm"]

axes[1].set_yticks([])
axes[0].set_yticks(np.arange(4) + 0.5, labels=labels, rotation=0)

for ax in axes.flatten():
    ax.set_xticks(
        np.arange(4) + 0.5, labels=labels, rotation=30, horizontalalignment="right"
    )


axes[0].set_title("Correlation")
axes[1].set_title("Partial correlation");

In [ ]:
# Set up the matplotlib figure
fig, axes = plt.subplots(
    ncols=2, width_ratios=[1, 2], figsize=(8, 3.5), constrained_layout=True
)

# Generate a custom diverging colormap
cmap = sns.diverging_palette(20, 230, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(
    partial_corr,
    mask=mask,
    cmap=cmap,
    annot=True,
    fmt=".1f",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    cbar=False,
    linewidths=0.5,
    ax=axes[0],
)

labels = ["CC Happening", "CC Anthropogenic (proxy)", "CC Worry", "Future gen harm"]

axes[0].set_yticks(np.arange(4) + 0.5, labels=labels, rotation=0)

axes[0].set_xticks(
    np.arange(4) + 0.5, labels=labels, rotation=30, horizontalalignment="right"
)


# ======== Network
adj = partial_corr.to_numpy()
adj[np.diag_indices_from(adj)] = 0
G = nx.from_numpy_array(adj)
node_labels = labels
edge_linewidths = {(u, v): z["weight"] * 4 for u, v, z in G.edges(data=True)}
edge_colours = [z["weight"] for u, v, z in G.edges(data=True)]
edge_labels = [f"{z['weight']:.2f}" for u, v, z in G.edges(data=True)]
layout = nx.circular_layout(G)


with ipx.style.context(
    [
        "hollow",
        {
            "vertex": {
                "linewidth": 0.5,
            },
            "edge": {
                "color": edge_colours,
                "alpha": 1,
                "cmap": "coolwarm_r",
                "norm": mcolors.Normalize(vmin=-1, vmax=1),
            },
        },
    ]
):
    network_artist = ipx.network(
        G,
        layout=layout,
        tension=1,
        edge_labels=edge_labels,
        node_labels=node_labels,
        edge_linewidth=edge_linewidths,
        edge_curved=True,
        # aspect="equal",
        margins=0.2,
        edge_label_bbox=dict(
            edgecolor="black",
            facecolor="white",
            linewidth=0.25,
            boxstyle="round,pad=0.3",
        ),
        edge_label_rotate=True,
        vertex_facecolor="white",
        vertex_zorder=3,
        ax=axes[1],
    )[0]
    fig.colorbar(
        network_artist.get_edges(),
        shrink=0.7,
        ax=axes[1],
    )

fig.suptitle("Partial correlation");

## Time-series with interaction: VAR

- $\mathbf{B}$: "Temporal network" of predictive effects from variables to variables in the next timestep.
- $\mathbf{K}$: "Contemporaneous network" of effects within a single timestep, after controlling for temporal effects.

Define later timestep as a regression problem on the earlier timestep:
$$
\begin{align*}
y_t &= \mathbf{B}y_{t-1} + \varepsilon_t \\
\varepsilon &\sim \mathcal{N}(\mathbf{0}, \mathbf{\Theta})
\end{align*}
$$

Then the contemporaneous network is given by $\mathbf{K} = \mathbf{\Theta}^{-1}$.

In [ ]:
# Isolate wave 1 (regressor) and wave 2 (response) data
resp_w1 = (
    resp_with_dk.filter(wave=1)
    .sort(by="participant_id")
    .drop("participant_id", "wave")
    .to_numpy()
)
resp_w2 = (
    resp_with_dk.filter(wave=2)
    .sort(by="participant_id")
    .drop("participant_id", "wave")
    .to_numpy()
)

# Regress wave 2 responses on wave 1 responses
X = resp_w1
B, *_ = np.linalg.lstsq(X, resp_w2)

# Calculate GGM; contemporaneous network
# 1. Variance-covariance matrix of residuals
w2_pred = (B @ X.T).T
resid = resp_w2 - w2_pred
theta = np.corrcoef(resid.T)

# 2. Invert and normalise
theta_inv = np.linalg.inv(theta)
denom = np.sqrt(np.outer(np.diag(theta_inv), np.diag(theta_inv)))
K = -(theta_inv / denom)
K[np.diag_indices_from(K)] = 1.0

In [ ]:
fig, axes = plt.subplots(
    ncols=2, width_ratios=[1, 2], figsize=(8, 3.5), constrained_layout=True
)

cmap = sns.diverging_palette(20, 230, as_cmap=True)

sns.heatmap(
    B,
    cmap=cmap,
    annot=True,
    fmt=".1f",
    center=0,
    square=True,
    linewidths=0.5,
    cbar=False,
    ax=axes[0],
)

labels = ["CC Happening", "CC Anthropogenic", "CC Worry", "Future gen harm"]


axes[0].set_yticks(np.arange(4) + 0.5, labels=labels, rotation=0)
axes[0].set_xticks(
    np.arange(4) + 0.5, labels=labels, rotation=30, horizontalalignment="right"
)
axes[0].set_xlabel("Wave 1")
axes[0].set_ylabel("Wave 2")


# ======= Network
adj = B.copy()
adj[abs(adj) < 0.1] = 0.0
G = nx.from_numpy_array(adj, create_using=nx.DiGraph)
node_labels = labels
edge_linewidths = {(u, v): z["weight"] * 4 for u, v, z in G.edges(data=True)}
edge_colours = [z["weight"] for u, v, z in G.edges(data=True)]
edge_labels = [f"{z['weight']:.2f}" for u, v, z in G.edges(data=True)]
layout = nx.circular_layout(G)
layout = np.array([[1.5, 0], [0, 1], [1.5, 1], [0, 0]])

with ipx.style.context(
    [
        "hollow",
        {
            "vertex": {
                "linewidth": 0.5,
            },
            "edge": {
                "color": edge_colours,
                "alpha": 1,
                "cmap": "coolwarm_r",
                "norm": mcolors.Normalize(vmin=-1, vmax=1),
                "looptension": 2.5,
                "arrow": {"width": 3},
            },
        },
    ]
):
    network_artist = ipx.network(
        G,
        layout=layout,
        tension=1,
        edge_labels=edge_labels,
        node_labels=node_labels,
        edge_linewidth=edge_linewidths,
        edge_curved=True,
        margins=0.2,
        edge_label_bbox=dict(
            edgecolor="black",
            facecolor="white",
            linewidth=0.25,
            boxstyle="round,pad=0.2",
        ),
        edge_label_rotate=True,
        vertex_facecolor="white",
        vertex_zorder=3,
        ax=axes[1],
    )[0]
    fig.colorbar(
        network_artist.get_edges(),
        shrink=0.7,
        ax=axes[1],
    )

fig.suptitle("VAR temporal network");

In [ ]:
fig, axes = plt.subplots(
    ncols=2, width_ratios=[1, 2], figsize=(8, 3.5), constrained_layout=True
)

cmap = sns.diverging_palette(20, 230, as_cmap=True)

mask = np.triu(np.ones_like(K, dtype=bool), k=1)

sns.heatmap(
    K,
    cmap=cmap,
    mask=mask,
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar=False,
    ax=axes[0],
)

labels = ["CC Happening", "CC Anthropogenic (proxy)", "CC Worry", "Future gen harm"]


axes[0].set_yticks(np.arange(4) + 0.5, labels=labels, rotation=0)
axes[0].set_xticks(
    np.arange(4) + 0.5, labels=labels, rotation=30, horizontalalignment="right"
)


# ======= Network
adj = K.copy()
adj[np.diag_indices_from(adj)] = 0
G = nx.from_numpy_array(adj)
node_labels = labels
edge_linewidths = {(u, v): z["weight"] * 8 for u, v, z in G.edges(data=True)}
edge_colours = [z["weight"] for u, v, z in G.edges(data=True)]
edge_labels = [f"{z['weight']:.2f}" for u, v, z in G.edges(data=True)]
layout = nx.circular_layout(G)
layout = np.array([[1, 0.2], [0, 1], [1, 0.8], [0, 0]])

with ipx.style.context(
    [
        "hollow",
        {
            "vertex": {
                "linewidth": 0.5,
            },
            "edge": {
                "color": edge_colours,
                "alpha": 1,
                "cmap": "coolwarm_r",
                "norm": mcolors.Normalize(vmin=-1, vmax=1),
            },
        },
    ]
):
    network_artist = ipx.network(
        G,
        layout=layout,
        tension=1,
        edge_labels=edge_labels,
        node_labels=node_labels,
        edge_linewidth=edge_linewidths,
        edge_curved=True,
        margins=0.2,
        edge_label_bbox=dict(
            edgecolor="black",
            facecolor="white",
            linewidth=0.25,
            boxstyle="round,pad=0.3",
        ),
        edge_label_rotate=True,
        vertex_facecolor="white",
        vertex_zorder=3,
        ax=axes[1],
    )[0]
    fig.colorbar(
        network_artist.get_edges(),
        shrink=0.7,
        ax=axes[1],
    )

fig.suptitle("VAR contemporaneous network");

## Within-question response transitions

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 12), constrained_layout=True)

pos = {
    "Climate change happening": np.array([[0, 0], [0.5, 0.87], [1, 0]]),
    "Climate change anthropogenic ('people should act')": np.array(
        [[1, 0], [1, 1 / 3], [0, 1 / 2], [1, 2 / 3], [1, 1]]
    ),
    "Climate change worry": np.array([[1, 0], [0, 0], [0, 1], [1, 1]]),
    "Future generation harm": np.array(
        [[0, 0], [1, 0], [1, 1 / 3], [1, 2 / 3], [1, 1]]
    ),
}

for i, (dimension, dimension_metadata) in enumerate(columns.items()):
    ax = axes[i // 2, i % 2]
    colname = dimension_metadata["colname"]
    responses = dimension_metadata["responses"]

    transition_probabilities = calculate_transition_probabilities(
        resp_with_dk, column=colname, start=1, end=2
    )

    if dimension_metadata["colname"].startswith("cc5_"):
        transition_probabilities = np.hstack(
            (np.zeros(5)[:, None], transition_probabilities)
        )

    # Calculate stationary distribution
    mu = calculate_stationary_distribution(transition_probabilities)

    transition_probabilities[transition_probabilities < 0.05] = 0

    G = nx.from_numpy_array(transition_probabilities, create_using=nx.DiGraph)
    edge_linewidth = {(u, v): 2.0 * z["weight"] for u, v, z in G.edges(data=True)}
    edge_labels = [
        f"{z['weight']:.2f}" if z["weight"] > 0.1 else None
        for u, v, z in G.edges(data=True)
    ]

    if dimension in pos:
        layout = pos[dimension]
    else:
        layout = nx.spring_layout(G, k=1)

    with ipx.style.context(
        [
            "hollow",
            {
                "vertex": {
                    "facecolor": mu,
                    "cmap": "Blues",
                    "alpha": 0.5,
                    "linewidth": 0.5,
                    # "norm": mcolors.Normalize(vmin=0, vmax=1),
                },
                "edge": {
                    "looptension": 3,
                    "loopmaxangle": 10,
                },
            },
        ]
    ):
        network_artist = ipx.network(
            G,
            layout=layout,
            tension=1,
            edge_labels=edge_labels,
            node_labels=responses,
            edge_linewidth=edge_linewidth,
            edge_curved=True,
            margins=0.2,
            aspect="equal",
            edge_label_bbox=dict(
                edgecolor="black",
                facecolor="white",
                boxstyle="round,pad=0.2",
                linewidth=0.25,
            ),
            edge_label_rotate=False,
            ax=ax,
        )[0]

    plt.colorbar(network_artist.get_vertices(), ax=ax, shrink=0.6)

    ax.set_title(dimension)

## Distance from equilibrium: Relative entropy

In [ ]:
plot_data = {"Question": [], "Relative entropy": []}

for i, (dimension, dimension_metadata) in enumerate(columns.items()):
    colname = dimension_metadata["colname"]

    transition_probabilities = calculate_transition_probabilities(
        resp_with_dk, column=colname, start=1, end=2
    )

    if dimension_metadata["colname"].startswith("cc5_"):
        transition_probabilities = np.hstack(
            (np.zeros(5)[:, None], transition_probabilities)
        )

    # Calculate stationary distribution
    mu = calculate_stationary_distribution(transition_probabilities)

    # Calculate wave 1 distribution
    props = calculate_proportion_response(resp_with_dk, column=colname, wave=1)

    # Calculate relative entropy (expected excess surprise when observing true props)
    plot_data["Question"].append(colname)
    plot_data["Relative entropy"].append(relative_entropy(props, mu))

plot_df = pl.DataFrame(plot_data)

sns.catplot(
    plot_df, x="Question", y="Relative entropy", kind="bar", aspect=1.3, height=2
);